In [2]:
# ============================================================
# STEP 5 — 2020 START-DATE S&P 500 UNIVERSE ROBUSTNESS ROUND
# Representative Market-State Stock Selection
#
# Implements Step 5 from plan.md:
#   Experiment 7.1 — Core representative-state comparison
#   Experiment 7.2 — Momentum benchmark
#   Experiment 7.3 — Random representative-date benchmark
#   Experiment 7.4 — Bootstrap confidence intervals
#
# Required input files in the same working directory:
#   project_skeleton.md
#   plan.md
#   sp500_constituents_2020_start.csv
#   experiment_input.zip
#   tda_step4_exp1_outputs.zip
#   tda_step4_exp2_outputs.zip
#   tda_step4_exp3_outputs.zip
#
# Output folder:
#   tda_step5_2020_universe_outputs/
#
# Output zip:
#   tda_step5_2020_universe_outputs.zip
#
# Important:
#   Variant 7A is used.
#   Universe fixed at 2020 start.
#   Pre-2020 prices are used only for 504-day lookback.
#   Evaluation starts only from the first valid rebalance on/after 2020-01-01.
# ============================================================

import os
import re
import json
import math
import time
import shutil
import zipfile
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


# ============================================================
# 0. CONFIG
# ============================================================

WORKDIR = Path.cwd()

REQUIRED_INPUT_FILES = [
    "project_skeleton.md",
    "plan.md",
    "sp500_constituents_2020_start.csv",
    "experiment_input.zip",
    "tda_step4_exp1_outputs.zip",
    "tda_step4_exp2_outputs.zip",
    "tda_step4_exp3_outputs.zip",
]

OUTPUT_DIR = WORKDIR / "tda_step5_2020_universe_outputs"
META_DIR = OUTPUT_DIR / "00_metadata"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_ZIP = WORKDIR / "tda_step5_2020_universe_outputs.zip"

ANCHOR_DATE = pd.Timestamp("2020-01-01")
EVALUATION_END_DATE = pd.Timestamp("2025-12-31")

PRICE_START_DATE = pd.Timestamp("2017-01-01")
PRICE_END_DATE_EXCLUSIVE = pd.Timestamp("2026-01-01")

W = 504
K_DEFAULT = 3
H_GRID = [21, 42, 63]
L_GRID = [0.20, 0.30, 0.40]

TRANSACTION_COST_RATE = 0.001
INITIAL_NAV = 1_000_000.0

RANDOM_SEED_BASE = 42
KMEANS_RANDOM_STATE = 42
KMEANS_N_INIT = 10

RANDOM_DATE_TRIALS = 100
BOOTSTRAP_SAMPLES = 1000
BOOTSTRAP_BLOCK_LENGTH = 21

YFINANCE_BATCH_SIZE = 80

MAPPER_COVER_INTERVALS = 10
MAPPER_OVERLAP = 0.30
MAPPER_DBSCAN_MIN_SAMPLES = 5
GLOBAL_DBSCAN_MIN_SAMPLES = 5
PCA_N_COMPONENTS_FOR_CLUSTERING = 10

np.random.seed(RANDOM_SEED_BASE)


# ============================================================
# 1. HARD INPUT CHECK
# ============================================================

missing_inputs = [f for f in REQUIRED_INPUT_FILES if not (WORKDIR / f).exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Missing required input file(s). Stop here. "
        f"Please provide: {missing_inputs}"
    )

print("All required input files are present.")
print("Working directory:", WORKDIR)


# ============================================================
# 2. CLEAN OUTPUT FOLDER
# ============================================================

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()


# ============================================================
# 3. UTILITY FUNCTIONS
# ============================================================

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def normalize_ticker(t):
    if pd.isna(t):
        return None
    t = str(t).strip()
    if not t:
        return None
    t = t.upper()
    t = t.replace(".", "-")
    return t


def safe_to_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)


def safe_to_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def zip_folder(folder_path, zip_path):
    folder_path = Path(folder_path)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in folder_path.rglob("*"):
            if p.is_file():
                zf.write(p, p.relative_to(folder_path.parent))


def extract_recursive_zip(zip_path, dest_dir):
    zip_path = Path(zip_path)
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)

    changed = True
    while changed:
        changed = False
        for nested_zip in list(dest_dir.rglob("*.zip")):
            nested_dest = nested_zip.with_suffix("")
            if not nested_dest.exists():
                nested_dest.mkdir(parents=True, exist_ok=True)
                try:
                    with zipfile.ZipFile(nested_zip, "r") as zf:
                        zf.extractall(nested_dest)
                    changed = True
                except zipfile.BadZipFile:
                    pass


def find_first_file(root, filename):
    root = Path(root)
    matches = list(root.rglob(filename))
    return matches[0] if matches else None


def df_to_markdown_simple(df):
    if df is None or len(df) == 0:
        return ""
    df2 = df.copy()
    cols = list(df2.columns)
    lines = []
    lines.append("| " + " | ".join(cols) + " |")
    lines.append("| " + " | ".join(["---"] * len(cols)) + " |")
    for _, row in df2.iterrows():
        vals = [str(row[c]) for c in cols]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


# ============================================================
# 4. METRICS — FIXED VERSION
# ============================================================

def sharpe_from_returns(rets):
    rets = pd.Series(rets).dropna()
    if len(rets) < 2:
        return np.nan
    sd = rets.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return np.nan
    return rets.mean() / sd * np.sqrt(252)


def annualized_return_from_total_return(total_return, n_daily_returns):
    if n_daily_returns is None or n_daily_returns <= 0:
        return np.nan
    if not np.isfinite(total_return):
        return np.nan
    if 1.0 + total_return <= 0:
        return np.nan
    return (1.0 + total_return) ** (252.0 / n_daily_returns) - 1.0


def max_drawdown_from_nav_with_initial(nav, initial_nav=None):
    if initial_nav is None:
        initial_nav = INITIAL_NAV

    nav = pd.Series(nav).dropna()
    if len(nav) == 0:
        return np.nan

    if isinstance(nav.index, pd.DatetimeIndex):
        initial_index = nav.index[0] - pd.Timedelta(nanoseconds=1)
    else:
        initial_index = -1

    nav_with_initial = pd.concat([
        pd.Series([initial_nav], index=[initial_index]),
        nav
    ])

    peak = nav_with_initial.cummax()
    dd = nav_with_initial / peak - 1.0
    return dd.min()


def compute_metrics(strategy_name, method, universe_label, h, l, k, W,
                    nav, daily_returns, turnover_list, nstocks_list,
                    fallback_count):
    nav = pd.Series(nav).dropna()
    daily_returns = pd.Series(daily_returns).dropna()

    if len(nav) < 1 or len(daily_returns) < 1:
        return {
            "strategy": strategy_name,
            "method": method,
            "universe_label": universe_label,
            "h": h,
            "l": l,
            "k": k,
            "W": W,
            "Cumulative Return": np.nan,
            "Annualized Return": np.nan,
            "Daily Sharpe": np.nan,
            "Daily Max Drawdown": np.nan,
            "Average Turnover": np.nan,
            "Average Number of Stocks": np.nan,
            "Rebalance Count": 0,
            "Fallback Count": fallback_count,
            "start_date": None,
            "evaluation_start_date": None,
            "evaluation_end_date": None,
        }

    cumulative_return = nav.iloc[-1] / INITIAL_NAV - 1.0

    return {
        "strategy": strategy_name,
        "method": method,
        "universe_label": universe_label,
        "h": h,
        "l": l,
        "k": k,
        "W": W,
        "Cumulative Return": cumulative_return,
        "Annualized Return": annualized_return_from_total_return(cumulative_return, len(daily_returns)),
        "Daily Sharpe": sharpe_from_returns(daily_returns),
        "Daily Max Drawdown": max_drawdown_from_nav_with_initial(nav, INITIAL_NAV),
        "Average Turnover": float(np.nanmean(turnover_list)) if len(turnover_list) else 0.0,
        "Average Number of Stocks": float(np.nanmean(nstocks_list)) if len(nstocks_list) else 0.0,
        "Rebalance Count": int(len(turnover_list)),
        "Fallback Count": int(fallback_count),
        "start_date": str(nav.index[0].date()) if isinstance(nav.index, pd.DatetimeIndex) else str(nav.index[0]),
        "evaluation_start_date": str(nav.index[0].date()) if isinstance(nav.index, pd.DatetimeIndex) else str(nav.index[0]),
        "evaluation_end_date": str(nav.index[-1].date()) if isinstance(nav.index, pd.DatetimeIndex) else str(nav.index[-1]),
    }


def metric_consistency(strategy_name, nav, daily_returns, tolerance=1e-8):
    nav = pd.Series(nav).dropna()
    daily_returns = pd.Series(daily_returns).dropna()

    if len(nav) < 1 or len(daily_returns) < 1:
        return {
            "strategy": strategy_name,
            "nav_cumulative_return": np.nan,
            "product_daily_return": np.nan,
            "absolute_error": np.nan,
            "tolerance": tolerance,
            "result": "INSUFFICIENT_DATA",
        }

    nav_cum = nav.iloc[-1] / INITIAL_NAV - 1.0
    prod_cum = (1.0 + daily_returns).prod() - 1.0
    err = abs(nav_cum - prod_cum)

    return {
        "strategy": strategy_name,
        "nav_cumulative_return": nav_cum,
        "product_daily_return": prod_cum,
        "absolute_error": err,
        "tolerance": tolerance,
        "result": "PASS" if err <= tolerance else "FAIL",
    }


# ============================================================
# 5. PORTFOLIO AND MODEL HELPERS
# ============================================================

def make_weights(tickers):
    tickers = list(dict.fromkeys([t for t in tickers if t is not None]))
    if len(tickers) == 0:
        return {}
    w = 1.0 / len(tickers)
    return {t: w for t in tickers}


def weights_turnover(old_weights, new_weights):
    keys = set(old_weights.keys()) | set(new_weights.keys())
    gross = sum(abs(new_weights.get(k, 0.0) - old_weights.get(k, 0.0)) for k in keys)
    return 0.5 * gross


def update_weights_after_returns(weights, day_returns):
    if not weights:
        return {}

    gross_values = {}
    portfolio_gross = 0.0

    for t, w in weights.items():
        r = day_returns.get(t, 0.0)
        if pd.isna(r):
            r = 0.0
        g = w * (1.0 + r)
        gross_values[t] = g
        portfolio_gross += g

    if portfolio_gross <= 0 or np.isnan(portfolio_gross):
        return weights

    return {t: g / portfolio_gross for t, g in gross_values.items()}


def eligible_tickers_at_idx(stock_returns, rb_idx, W):
    train = stock_returns.iloc[rb_idx - W: rb_idx]
    return train.columns[train.notna().all(axis=0)].tolist()


def standardize_train_matrix(train):
    scaler = StandardScaler()
    X = scaler.fit_transform(train.values)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


def rank_top_l_on_dates(train_returns, representative_dates, l):
    selected = set()

    for d in representative_dates:
        if d not in train_returns.index:
            continue

        row = train_returns.loc[d].dropna()
        if len(row) == 0:
            continue

        n_select = max(1, int(math.ceil(len(row) * l)))
        selected.update(row.sort_values(ascending=False).head(n_select).index.tolist())

    return sorted(selected)


def cluster_representative_dates_from_labels(X, dates, labels, k):
    dates = pd.Index(dates)
    labels = np.asarray(labels)

    valid_labels = [lab for lab in sorted(set(labels)) if lab != -1]
    clusters = []

    for lab in valid_labels:
        idx = np.where(labels == lab)[0]
        if len(idx) > 0:
            clusters.append((lab, idx))

    clusters = sorted(clusters, key=lambda x: len(x[1]), reverse=True)[:k]

    reps = []
    for lab, idx in clusters:
        Xc = X[idx]
        centroid = Xc.mean(axis=0)
        dist = np.linalg.norm(Xc - centroid, axis=1)
        chosen_local = int(np.argmin(dist))
        reps.append(dates[idx[chosen_local]])

    return list(dict.fromkeys(reps))


def auto_dbscan_labels(X, min_samples=5, random_state=42, fallback_k=3):
    n = X.shape[0]
    if n == 0:
        return np.array([])
    if n < max(2, min_samples):
        return np.zeros(n, dtype=int)

    nn_k = min(max(2, min_samples), n)

    try:
        nbrs = NearestNeighbors(n_neighbors=nn_k).fit(X)
        distances, _ = nbrs.kneighbors(X)
        kth = distances[:, -1]
        eps = np.nanpercentile(kth, 70)

        if not np.isfinite(eps) or eps <= 0:
            eps = np.nanmedian(kth)
        if not np.isfinite(eps) or eps <= 0:
            eps = 0.5

        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
    except Exception:
        labels = np.full(n, -1)

    non_noise = [x for x in set(labels) if x != -1]

    if len(non_noise) == 0:
        kk = min(max(1, fallback_k), n)
        labels = KMeans(
            n_clusters=kk,
            random_state=random_state,
            n_init=KMEANS_N_INIT
        ).fit_predict(X)

    return labels


def select_pca_kmeans(train_returns, l, k, random_state=42):
    train_returns = train_returns.dropna(axis=1, how="any")

    if train_returns.shape[1] < max(2, k) or train_returns.shape[0] < k:
        return [], [], True

    X = standardize_train_matrix(train_returns)
    ncomp = min(PCA_N_COMPONENTS_FOR_CLUSTERING, X.shape[0], X.shape[1])
    Z = PCA(n_components=ncomp, random_state=random_state).fit_transform(X)

    kk = min(k, Z.shape[0])
    labels = KMeans(
        n_clusters=kk,
        random_state=random_state,
        n_init=KMEANS_N_INIT
    ).fit_predict(Z)

    reps = cluster_representative_dates_from_labels(Z, train_returns.index, labels, k=kk)

    if len(reps) == 0:
        return [], [], True

    selected = rank_top_l_on_dates(train_returns, reps, l)
    return selected, reps, len(selected) == 0


def select_global_dbscan(train_returns, l, k, random_state=42):
    train_returns = train_returns.dropna(axis=1, how="any")

    if train_returns.shape[1] < max(2, k) or train_returns.shape[0] < k:
        return [], [], True

    X = standardize_train_matrix(train_returns)
    ncomp = min(PCA_N_COMPONENTS_FOR_CLUSTERING, X.shape[0], X.shape[1])
    Z = PCA(n_components=ncomp, random_state=random_state).fit_transform(X)

    labels = auto_dbscan_labels(
        Z,
        min_samples=GLOBAL_DBSCAN_MIN_SAMPLES,
        random_state=random_state,
        fallback_k=k
    )

    reps = cluster_representative_dates_from_labels(Z, train_returns.index, labels, k=k)

    if len(reps) == 0:
        return [], [], True

    selected = rank_top_l_on_dates(train_returns, reps, l)
    return selected, reps, len(selected) == 0


def mapper_nodes_from_pc1(X, pc1, n_intervals=10, overlap=0.30, min_samples=5):
    n = len(pc1)
    if n == 0:
        return []

    pc_min = float(np.nanmin(pc1))
    pc_max = float(np.nanmax(pc1))

    if not np.isfinite(pc_min) or not np.isfinite(pc_max) or pc_min == pc_max:
        return [np.arange(n)]

    span = pc_max - pc_min
    interval_width = span / (n_intervals - (n_intervals - 1) * overlap)
    step = interval_width * (1 - overlap)

    nodes = []
    seen = set()

    for i in range(n_intervals):
        lo = pc_min + i * step
        hi = lo + interval_width

        if i == n_intervals - 1:
            mask = (pc1 >= lo) & (pc1 <= hi)
        else:
            mask = (pc1 >= lo) & (pc1 < hi)

        idx = np.where(mask)[0]

        if len(idx) == 0:
            continue

        if len(idx) < max(2, min_samples):
            key = tuple(sorted(idx.tolist()))
            if key not in seen:
                nodes.append(idx)
                seen.add(key)
            continue

        local_X = X[idx]
        labels = auto_dbscan_labels(
            local_X,
            min_samples=min_samples,
            random_state=KMEANS_RANDOM_STATE,
            fallback_k=1
        )

        for lab in sorted(set(labels)):
            if lab == -1:
                continue

            local_idx = idx[np.where(labels == lab)[0]]

            if len(local_idx) == 0:
                continue

            key = tuple(sorted(local_idx.tolist()))
            if key not in seen:
                nodes.append(local_idx)
                seen.add(key)

    return nodes


def select_mapper(train_returns, l, k, random_state=42):
    train_returns = train_returns.dropna(axis=1, how="any")

    if train_returns.shape[1] < max(2, k) or train_returns.shape[0] < k:
        return [], [], True

    X = standardize_train_matrix(train_returns)
    pc1 = PCA(n_components=1, random_state=random_state).fit_transform(X).ravel()

    nodes = mapper_nodes_from_pc1(
        X=X,
        pc1=pc1,
        n_intervals=MAPPER_COVER_INTERVALS,
        overlap=MAPPER_OVERLAP,
        min_samples=MAPPER_DBSCAN_MIN_SAMPLES
    )

    nodes = sorted(nodes, key=lambda idx: len(idx), reverse=True)[:k]

    reps = []
    for idx in nodes:
        Xc = X[idx]
        centroid = Xc.mean(axis=0)
        dist = np.linalg.norm(Xc - centroid, axis=1)
        chosen = idx[int(np.argmin(dist))]
        reps.append(train_returns.index[chosen])

    reps = list(dict.fromkeys(reps))

    if len(reps) == 0:
        return [], [], True

    selected = rank_top_l_on_dates(train_returns, reps, l)
    return selected, reps, len(selected) == 0


def select_momentum(train_returns, l, momentum_lookback):
    train_returns = train_returns.dropna(axis=1, how="any")

    if train_returns.shape[1] == 0 or train_returns.shape[0] < momentum_lookback:
        return [], True

    trailing = train_returns.iloc[-momentum_lookback:]
    score = (1.0 + trailing).prod(axis=0) - 1.0
    score = score.dropna()

    if len(score) == 0:
        return [], True

    n_select = max(1, int(math.ceil(len(score) * l)))
    selected = score.sort_values(ascending=False).head(n_select).index.tolist()

    return selected, False


def select_random_dates(train_returns, l, k, rng):
    train_returns = train_returns.dropna(axis=1, how="any")

    if train_returns.shape[1] < max(2, k) or train_returns.shape[0] < k:
        return [], [], True

    chosen_positions = rng.choice(np.arange(train_returns.shape[0]), size=k, replace=False)
    reps = train_returns.index[chosen_positions].tolist()
    selected = rank_top_l_on_dates(train_returns, reps, l)

    return selected, reps, len(selected) == 0


def get_clean_rebalance_indices(stock_returns, h, W, anchor_date, k=3):
    idx_anchor_candidates = np.where(stock_returns.index >= anchor_date)[0]

    if len(idx_anchor_candidates) == 0:
        raise RuntimeError("No stock return dates at or after anchor date.")

    first_anchor_idx = int(idx_anchor_candidates[0])
    first_possible_idx = max(W, first_anchor_idx)

    raw_rebalance_indices = list(range(first_possible_idx, len(stock_returns.index), h))

    if len(raw_rebalance_indices) == 0:
        raise RuntimeError(f"No rebalance indices available for h={h}.")

    raw_first_rebalance_date = stock_returns.index[raw_rebalance_indices[0]]

    clean_start_pos = None
    skipped = 0

    for pos, rb_idx in enumerate(raw_rebalance_indices):
        eligible = eligible_tickers_at_idx(stock_returns, rb_idx, W)

        if len(eligible) >= max(2, k):
            clean_start_pos = pos
            break

        skipped += 1

    if clean_start_pos is None:
        raise RuntimeError(
            f"No valid rebalance date found for h={h}; "
            f"minimum eligible count required = {max(2, k)}."
        )

    clean_rebalance_indices = raw_rebalance_indices[clean_start_pos:]
    clean_evaluation_start_date = stock_returns.index[clean_rebalance_indices[0]]

    if clean_evaluation_start_date < anchor_date:
        raise RuntimeError(
            f"Clean evaluation start date {clean_evaluation_start_date} is before anchor date {anchor_date}."
        )

    return {
        "h": h,
        "raw_rebalance_indices": raw_rebalance_indices,
        "clean_rebalance_indices": clean_rebalance_indices,
        "raw_first_rebalance_date": raw_first_rebalance_date,
        "clean_evaluation_start_date": clean_evaluation_start_date,
        "skipped_initial_rebalance_count": skipped,
        "reason_for_skipping_initial_invalid_rebalances": (
            f"Initial rebalance dates are skipped until eligible_count >= max(2, k) = {max(2, k)} "
            "using a complete non-missing 504-day lookback window."
        ),
    }


# ============================================================
# 6. EXTRACT INPUT ZIP FILES AND VERIFY METADATA
# ============================================================

EXTRACT_DIR = OUTPUT_DIR / "_extracted_inputs"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

for z in [
    "experiment_input.zip",
    "tda_step4_exp1_outputs.zip",
    "tda_step4_exp2_outputs.zip",
    "tda_step4_exp3_outputs.zip",
]:
    dest = EXTRACT_DIR / Path(z).stem
    extract_recursive_zip(WORKDIR / z, dest)

step1_meta = find_first_file(EXTRACT_DIR / "experiment_input", "step1_experiment_metadata.json")

if step1_meta is None:
    raise FileNotFoundError(
        "experiment_input.zip was extracted, but step1_experiment_metadata.json was not found. Stop here."
    )

with open(step1_meta, "r", encoding="utf-8") as f:
    step1_metadata = json.load(f)

if "initial_nav" in step1_metadata:
    try:
        INITIAL_NAV = float(step1_metadata["initial_nav"])
    except Exception:
        pass

print("Found step1_experiment_metadata.json:", step1_meta)
print("INITIAL_NAV used:", INITIAL_NAV)


# ============================================================
# 7. READ PROJECT FILES AND 2020 UNIVERSE
# ============================================================

project_skeleton_text = (WORKDIR / "project_skeleton.md").read_text(encoding="utf-8")
plan_text = (WORKDIR / "plan.md").read_text(encoding="utf-8")

write_text(META_DIR / "project_skeleton.md", project_skeleton_text)
write_text(META_DIR / "plan.md", plan_text)

universe_raw = pd.read_csv(WORKDIR / "sp500_constituents_2020_start.csv")

possible_ticker_cols = ["ticker", "Ticker", "symbol", "Symbol"]
ticker_col = None

for c in possible_ticker_cols:
    if c in universe_raw.columns:
        ticker_col = c
        break

if ticker_col is None:
    raise ValueError(
        "sp500_constituents_2020_start.csv must contain one of these ticker columns: "
        f"{possible_ticker_cols}"
    )

universe_raw["raw_ticker"] = universe_raw[ticker_col].astype(str).str.strip()
universe_raw["normalized_ticker"] = universe_raw["raw_ticker"].apply(normalize_ticker)

universe_clean = (
    universe_raw
    .dropna(subset=["normalized_ticker"])
    .drop_duplicates(subset=["normalized_ticker"])
    .copy()
)

tickers = universe_clean["normalized_ticker"].tolist()

if len(tickers) < 100:
    raise RuntimeError(
        f"Universe has only {len(tickers)} normalized unique tickers. This is unexpectedly small."
    )

safe_to_csv(universe_clean, META_DIR / "step5_2020_universe_tickers.csv", index=False)

print("Raw ticker count:", len(universe_raw))
print("Normalized unique ticker count:", len(tickers))


# ============================================================
# 8. DOWNLOAD STOCK AND INDEX DATA
# ============================================================

try:
    import yfinance as yf
except Exception as e:
    raise ImportError("yfinance is required. Please install it with: pip install yfinance") from e


def extract_close_from_yfinance_download(data):
    if data is None or len(data) == 0:
        return pd.DataFrame()

    if isinstance(data.columns, pd.MultiIndex):
        level0 = data.columns.get_level_values(0)

        if "Close" in level0:
            close = data["Close"].copy()
        elif "Adj Close" in level0:
            close = data["Adj Close"].copy()
        else:
            raise ValueError("Could not find Close or Adj Close in yfinance MultiIndex columns.")
    else:
        if "Close" in data.columns:
            close = data[["Close"]].copy()
        elif "Adj Close" in data.columns:
            close = data[["Adj Close"]].copy()
        else:
            raise ValueError("Could not find Close or Adj Close in yfinance columns.")

    if isinstance(close, pd.Series):
        close = close.to_frame()

    close.index = pd.to_datetime(close.index)
    close = close.sort_index()

    return close


def download_prices_yfinance(tickers, start, end, batch_size=80, auto_adjust=True):
    all_prices = []
    failed_batches = []

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i + batch_size]
        print(f"Downloading batch {i // batch_size + 1}: {len(batch)} tickers")

        try:
            data = yf.download(
                tickers=batch,
                start=str(start.date()),
                end=str(end.date()),
                auto_adjust=auto_adjust,
                progress=False,
                group_by="column",
                threads=True,
            )

            close = extract_close_from_yfinance_download(data)

            if len(batch) == 1 and close.shape[1] == 1:
                close.columns = [batch[0]]

            all_prices.append(close)

        except Exception as e:
            failed_batches.append({
                "batch_start": i,
                "batch_end": i + len(batch) - 1,
                "tickers": ",".join(batch),
                "error": repr(e),
            })

        time.sleep(0.25)

    if all_prices:
        prices = pd.concat(all_prices, axis=1)
        prices = prices.loc[:, ~prices.columns.duplicated()]
    else:
        prices = pd.DataFrame()

    return prices, pd.DataFrame(failed_batches)


stock_prices, failed_batches_df = download_prices_yfinance(
    tickers=tickers,
    start=PRICE_START_DATE,
    end=PRICE_END_DATE_EXCLUSIVE,
    batch_size=YFINANCE_BATCH_SIZE,
    auto_adjust=True,
)

if stock_prices.empty:
    raise RuntimeError("No stock prices downloaded from yfinance.")

stock_prices.columns = [normalize_ticker(c) for c in stock_prices.columns]
stock_prices = stock_prices.loc[:, [c for c in stock_prices.columns if c is not None]]
stock_prices = stock_prices.sort_index()
stock_prices = stock_prices.loc[:, ~stock_prices.columns.duplicated()]

stock_returns = stock_prices.pct_change(fill_method=None)

available_return_cols = stock_returns.columns[stock_returns.notna().any(axis=0)].tolist()
stock_returns = stock_returns[available_return_cols]
stock_prices = stock_prices[[c for c in stock_prices.columns if c in available_return_cols]]

stock_returns = stock_returns.loc[
    (stock_returns.index >= PRICE_START_DATE) &
    (stock_returns.index <= EVALUATION_END_DATE)
]

stock_prices = stock_prices.reindex(stock_returns.index)

if stock_returns.index.max() < ANCHOR_DATE:
    raise RuntimeError("Downloaded stock returns end before anchor date. Cannot run Step 5.")

available_set = set(stock_returns.columns)

missing_rows = []

for _, row in universe_clean.iterrows():
    raw_t = row["raw_ticker"]
    norm_t = row["normalized_ticker"]

    if norm_t not in available_set:
        status = "missing_or_unavailable"
        reason = "not_returned_by_yfinance_or_removed_after_validation"
    else:
        s = stock_returns[norm_t]
        if s.notna().sum() == 0:
            status = "missing_or_unavailable"
            reason = "all_returns_missing"
        else:
            status = "available"
            reason = "available"

    missing_rows.append({
        "raw_ticker": raw_t,
        "normalized_ticker": norm_t,
        "status": status,
        "reason": reason,
    })

missing_unavailable_df = pd.DataFrame(missing_rows)

raw_ticker_count = len(universe_raw)
normalized_unique_ticker_count = len(universe_clean)
available_ticker_count = int((missing_unavailable_df["status"] == "available").sum())
missing_unavailable_ticker_count = int((missing_unavailable_df["status"] == "missing_or_unavailable").sum())

safe_to_csv(stock_prices, META_DIR / "downloaded_stock_prices_yfinance_auto_adjust_true.csv", index=True)
safe_to_csv(stock_returns, META_DIR / "downloaded_stock_returns_yfinance_auto_adjust_true.csv", index=True)
safe_to_csv(failed_batches_df, META_DIR / "failed_yfinance_batches.csv", index=False)
safe_to_csv(missing_unavailable_df, META_DIR / "missing_or_unavailable_universe_tickers.csv", index=False)

print("Downloaded stock price matrix:", stock_prices.shape)
print("Downloaded stock return matrix:", stock_returns.shape)
print("Available ticker count:", available_ticker_count)
print("Missing/unavailable ticker count:", missing_unavailable_ticker_count)


sp500_data = yf.download(
    tickers="^GSPC",
    start=str(PRICE_START_DATE.date()),
    end=str(PRICE_END_DATE_EXCLUSIVE.date()),
    auto_adjust=True,
    progress=False,
    group_by="column",
    threads=True,
)

sp500_prices = extract_close_from_yfinance_download(sp500_data)

if sp500_prices.empty:
    raise RuntimeError("Could not download ^GSPC benchmark from yfinance.")

if sp500_prices.shape[1] > 1:
    sp500_prices = sp500_prices.iloc[:, [0]]

sp500_prices.columns = ["^GSPC"]
sp500_prices = sp500_prices.sort_index()
sp500_returns = sp500_prices["^GSPC"].pct_change(fill_method=None).reindex(stock_returns.index)

safe_to_csv(sp500_prices, META_DIR / "downloaded_sp500_index_yfinance_auto_adjust_true.csv", index=True)

print("Downloaded S&P 500 benchmark:", sp500_prices.shape)


# ============================================================
# 9. BACKTEST ENGINE
# ============================================================

def backtest_strategy(stock_returns, strategy_type, method_label, h, l=None, k=3, W=504,
                      momentum_lookback=None, random_seed=None):
    rb_info = get_clean_rebalance_indices(stock_returns, h=h, W=W, anchor_date=ANCHOR_DATE, k=k)
    rb_indices = rb_info["clean_rebalance_indices"]
    evaluation_start_idx = rb_indices[0]
    evaluation_start_date = stock_returns.index[evaluation_start_idx]

    if evaluation_start_date < ANCHOR_DATE:
        raise RuntimeError(
            f"{strategy_type} h={h} starts before anchor date: {evaluation_start_date}"
        )

    rb_set = set(rb_indices)
    rng = np.random.default_rng(random_seed if random_seed is not None else RANDOM_SEED_BASE)

    nav_values = []
    ret_values = []
    nav_dates = []

    rebalance_log_rows = []
    selection_log_rows = []

    nav = INITIAL_NAV
    current_weights = {}
    turnover_list = []
    nstocks_list = []
    fallback_count = 0

    last_idx = len(stock_returns.index) - 1

    for day_idx in range(evaluation_start_idx, last_idx + 1):
        date = stock_returns.index[day_idx]
        nav_before_day = nav

        if day_idx in rb_set:
            train_returns = stock_returns.iloc[day_idx - W: day_idx].copy()
            eligible = train_returns.columns[train_returns.notna().all(axis=0)].tolist()
            eligible_count = len(eligible)

            if eligible_count < max(2, k):
                raise RuntimeError(
                    f"Invalid clean rebalance encountered: {date.date()}, "
                    f"eligible_count={eligible_count}, required={max(2, k)}"
                )

            train_eligible = train_returns[eligible].copy()
            selected = []
            reps = []
            fallback_used = False

            if strategy_type == "PCA-KMeans":
                selected, reps, fallback_used = select_pca_kmeans(
                    train_eligible, l=l, k=k, random_state=KMEANS_RANDOM_STATE
                )
            elif strategy_type == "Mapper":
                selected, reps, fallback_used = select_mapper(
                    train_eligible, l=l, k=k, random_state=KMEANS_RANDOM_STATE
                )
            elif strategy_type == "Global DBSCAN":
                selected, reps, fallback_used = select_global_dbscan(
                    train_eligible, l=l, k=k, random_state=KMEANS_RANDOM_STATE
                )
            elif strategy_type == "Equal Weight Universe":
                selected = eligible
                reps = []
                fallback_used = False
            elif strategy_type == "Momentum":
                selected, fallback_used = select_momentum(
                    train_eligible, l=l, momentum_lookback=momentum_lookback
                )
                reps = []
            elif strategy_type == "Random Representative-Date":
                selected, reps, fallback_used = select_random_dates(
                    train_eligible, l=l, k=k, rng=rng
                )
            else:
                raise ValueError(f"Unknown strategy_type: {strategy_type}")

            if len(selected) == 0:
                selected = eligible
                fallback_used = True

            if fallback_used:
                fallback_count += 1

            new_weights = make_weights(selected)
            turnover = weights_turnover(current_weights, new_weights)
            transaction_cost = nav * turnover * TRANSACTION_COST_RATE
            nav = nav - transaction_cost
            current_weights = new_weights

            turnover_list.append(turnover)
            nstocks_list.append(len(selected))

            rebalance_log_rows.append({
                "date": str(date.date()),
                "strategy": method_label,
                "strategy_method": strategy_type,
                "h": h,
                "l": l,
                "k": k,
                "W": W,
                "momentum_lookback": momentum_lookback,
                "eligible_count": eligible_count,
                "selected_stock_count": len(selected),
                "representative_dates": ",".join([str(pd.Timestamp(x).date()) for x in reps]) if reps else "",
                "fallback_used": bool(fallback_used),
                "turnover": turnover,
                "transaction_cost": transaction_cost,
                "nav_after_rebalance_cost_before_day_return": nav,
            })

            for s in selected:
                selection_log_rows.append({
                    "date": str(date.date()),
                    "strategy": method_label,
                    "strategy_method": strategy_type,
                    "h": h,
                    "l": l,
                    "k": k,
                    "W": W,
                    "momentum_lookback": momentum_lookback,
                    "selected_ticker": s,
                    "selected_weight": current_weights.get(s, np.nan),
                    "fallback_used": bool(fallback_used),
                })

        day_rets = stock_returns.iloc[day_idx].to_dict()

        if current_weights:
            portfolio_day_return = 0.0

            for t, w in current_weights.items():
                r = day_rets.get(t, 0.0)
                if pd.isna(r):
                    r = 0.0
                portfolio_day_return += w * r

            nav = nav * (1.0 + portfolio_day_return)
            current_weights = update_weights_after_returns(current_weights, day_rets)
        else:
            portfolio_day_return = 0.0

        daily_return_after_cost = nav / nav_before_day - 1.0

        nav_dates.append(date)
        nav_values.append(nav)
        ret_values.append(daily_return_after_cost)

    nav_series = pd.Series(nav_values, index=pd.DatetimeIndex(nav_dates), name=method_label)
    ret_series = pd.Series(ret_values, index=pd.DatetimeIndex(nav_dates), name=method_label)

    metrics_row = compute_metrics(
        strategy_name=method_label,
        method=strategy_type,
        universe_label="S&P 500 2020-start fixed universe",
        h=h,
        l=l,
        k=k,
        W=W,
        nav=nav_series,
        daily_returns=ret_series,
        turnover_list=turnover_list,
        nstocks_list=nstocks_list,
        fallback_count=fallback_count,
    )

    metrics_row.update({
        "raw_first_rebalance_date": str(rb_info["raw_first_rebalance_date"].date()),
        "clean_evaluation_start_date": str(rb_info["clean_evaluation_start_date"].date()),
        "skipped_initial_rebalance_count": int(rb_info["skipped_initial_rebalance_count"]),
        "reason_for_skipping_initial_invalid_rebalances": rb_info["reason_for_skipping_initial_invalid_rebalances"],
    })

    return {
        "strategy": method_label,
        "strategy_type": strategy_type,
        "h": h,
        "l": l,
        "k": k,
        "W": W,
        "nav": nav_series,
        "daily_returns": ret_series,
        "metrics": metrics_row,
        "rebalance_log": pd.DataFrame(rebalance_log_rows),
        "selection_log": pd.DataFrame(selection_log_rows),
        "rb_info": rb_info,
    }


def backtest_sp500_benchmark(stock_returns, sp500_returns, h, W=504, k=3):
    rb_info = get_clean_rebalance_indices(stock_returns, h=h, W=W, anchor_date=ANCHOR_DATE, k=k)
    start_idx = rb_info["clean_rebalance_indices"][0]
    start_date = stock_returns.index[start_idx]

    idx = stock_returns.index[start_idx:]
    r = sp500_returns.reindex(idx).fillna(0.0)

    nav = INITIAL_NAV * (1.0 + r).cumprod()
    daily = r.copy()

    strategy_name = f"S&P 500 Index Benchmark | h={h}"

    rb_log = pd.DataFrame([{
        "date": str(start_date.date()),
        "strategy": strategy_name,
        "strategy_method": "S&P 500 Index Benchmark",
        "h": h,
        "l": np.nan,
        "k": k,
        "W": W,
        "momentum_lookback": np.nan,
        "eligible_count": np.nan,
        "selected_stock_count": np.nan,
        "representative_dates": "",
        "fallback_used": False,
        "turnover": 0.0,
        "transaction_cost": 0.0,
        "nav_after_rebalance_cost_before_day_return": INITIAL_NAV,
    }])

    metrics_row = compute_metrics(
        strategy_name=strategy_name,
        method="S&P 500 Index Benchmark",
        universe_label="S&P 500 2020-start fixed universe",
        h=h,
        l=np.nan,
        k=k,
        W=W,
        nav=nav,
        daily_returns=daily,
        turnover_list=[],
        nstocks_list=[],
        fallback_count=0,
    )

    metrics_row.update({
        "raw_first_rebalance_date": str(rb_info["raw_first_rebalance_date"].date()),
        "clean_evaluation_start_date": str(rb_info["clean_evaluation_start_date"].date()),
        "skipped_initial_rebalance_count": int(rb_info["skipped_initial_rebalance_count"]),
        "reason_for_skipping_initial_invalid_rebalances": rb_info["reason_for_skipping_initial_invalid_rebalances"],
    })

    return {
        "strategy": strategy_name,
        "strategy_type": "S&P 500 Index Benchmark",
        "h": h,
        "l": np.nan,
        "k": k,
        "W": W,
        "nav": nav.rename(strategy_name),
        "daily_returns": daily.rename(strategy_name),
        "metrics": metrics_row,
        "rebalance_log": rb_log,
        "selection_log": pd.DataFrame(),
        "rb_info": rb_info,
    }


# ============================================================
# 10. EXPERIMENT 7.1 — CORE REPRESENTATIVE-STATE COMPARISON
# ============================================================

core_results = []

for h in H_GRID:
    for l in L_GRID:
        for method in ["PCA-KMeans", "Mapper", "Global DBSCAN"]:
            strategy_name = f"{method} Representative-State, 2020-start universe | h={h} | l={l:.2f} | k={K_DEFAULT}"
            print("Running core:", strategy_name)

            res = backtest_strategy(
                stock_returns=stock_returns,
                strategy_type=method,
                method_label=strategy_name,
                h=h,
                l=l,
                k=K_DEFAULT,
                W=W,
            )

            core_results.append(res)

    ew_name = f"Equal Weight Universe, 2020-start universe | h={h}"
    print("Running core:", ew_name)

    core_results.append(
        backtest_strategy(
            stock_returns=stock_returns,
            strategy_type="Equal Weight Universe",
            method_label=ew_name,
            h=h,
            l=np.nan,
            k=K_DEFAULT,
            W=W,
        )
    )

    print("Running core: S&P 500 h=", h)

    core_results.append(
        backtest_sp500_benchmark(
            stock_returns=stock_returns,
            sp500_returns=sp500_returns,
            h=h,
            W=W,
            k=K_DEFAULT,
        )
    )

core_metrics = pd.DataFrame([r["metrics"] for r in core_results])
core_daily_nav = pd.concat([r["nav"] for r in core_results], axis=1)
core_daily_returns = pd.concat([r["daily_returns"] for r in core_results], axis=1)
core_rebalance_log = pd.concat([r["rebalance_log"] for r in core_results], ignore_index=True)

core_selection_logs = [r["selection_log"] for r in core_results if not r["selection_log"].empty]
core_selection_log = pd.concat(core_selection_logs, ignore_index=True) if core_selection_logs else pd.DataFrame()

safe_to_csv(core_metrics, OUTPUT_DIR / "step5_2020_core_metrics.csv", index=False)
safe_to_csv(core_daily_nav, OUTPUT_DIR / "step5_2020_core_daily_nav.csv", index=True)
safe_to_csv(core_daily_returns, OUTPUT_DIR / "step5_2020_core_daily_returns.csv", index=True)
safe_to_csv(core_rebalance_log, OUTPUT_DIR / "step5_2020_core_rebalance_log.csv", index=False)
safe_to_csv(core_selection_log, OUTPUT_DIR / "step5_2020_core_selection_log.csv", index=False)


# ============================================================
# 11. BALANCED SCORE RANKING FOR CORE
# ============================================================

ranking = core_metrics.copy()

ranking["rank_annualized_return"] = ranking["Annualized Return"].rank(ascending=False, method="average")
ranking["rank_daily_sharpe"] = ranking["Daily Sharpe"].rank(ascending=False, method="average")
ranking["rank_daily_max_drawdown"] = ranking["Daily Max Drawdown"].rank(ascending=False, method="average")
ranking["rank_average_turnover"] = ranking["Average Turnover"].rank(ascending=True, method="average")

ranking["Balanced Score"] = (
    0.35 * ranking["rank_annualized_return"]
    + 0.35 * ranking["rank_daily_sharpe"]
    + 0.20 * ranking["rank_daily_max_drawdown"]
    + 0.10 * ranking["rank_average_turnover"]
)

ranking = ranking.sort_values("Balanced Score", ascending=True).reset_index(drop=True)
ranking["Balanced Score Rank"] = np.arange(1, len(ranking) + 1)

safe_to_csv(ranking, OUTPUT_DIR / "step5_2020_core_balanced_score_ranking.csv", index=False)

balanced_score_formula = {
    "Balanced Score": "0.35*rank(Annualized Return) + 0.35*rank(Daily Sharpe) + 0.20*rank(Daily Max Drawdown) + 0.10*rank(Average Turnover)",
    "ranking_rules": {
        "Annualized Return": "higher is better",
        "Daily Sharpe": "higher is better",
        "Daily Max Drawdown": "less negative / higher is better",
        "Average Turnover": "lower is better",
    },
}

safe_to_json(balanced_score_formula, OUTPUT_DIR / "step5_2020_balanced_score_formula.json")


# ============================================================
# 12. EXPERIMENT 7.2 — MOMENTUM BENCHMARK
# ============================================================

momentum_results = []
momentum_lookbacks = [21, 63, 126]

for h in H_GRID:
    for l in L_GRID:
        for mlook in momentum_lookbacks:
            strategy_name = f"Momentum {mlook}D, 2020-start universe | h={h} | l={l:.2f}"
            print("Running momentum:", strategy_name)

            res = backtest_strategy(
                stock_returns=stock_returns,
                strategy_type="Momentum",
                method_label=strategy_name,
                h=h,
                l=l,
                k=K_DEFAULT,
                W=W,
                momentum_lookback=mlook,
            )

            momentum_results.append(res)

momentum_metrics = pd.DataFrame([r["metrics"] for r in momentum_results])
momentum_daily_nav = pd.concat([r["nav"] for r in momentum_results], axis=1)
momentum_daily_returns = pd.concat([r["daily_returns"] for r in momentum_results], axis=1)
momentum_rebalance_log = pd.concat([r["rebalance_log"] for r in momentum_results], ignore_index=True)

momentum_selection_logs = [r["selection_log"] for r in momentum_results if not r["selection_log"].empty]
momentum_selection_log = pd.concat(momentum_selection_logs, ignore_index=True) if momentum_selection_logs else pd.DataFrame()

safe_to_csv(momentum_metrics, OUTPUT_DIR / "step5_2020_momentum_metrics.csv", index=False)
safe_to_csv(momentum_daily_nav, OUTPUT_DIR / "step5_2020_momentum_daily_nav.csv", index=True)
safe_to_csv(momentum_daily_returns, OUTPUT_DIR / "step5_2020_momentum_daily_returns.csv", index=True)
safe_to_csv(momentum_rebalance_log, OUTPUT_DIR / "step5_2020_momentum_rebalance_log.csv", index=False)
safe_to_csv(momentum_selection_log, OUTPUT_DIR / "step5_2020_momentum_selection_log.csv", index=False)

best_core_by_method = (
    ranking[
        ranking["method"].isin([
            "PCA-KMeans",
            "Mapper",
            "Global DBSCAN",
            "Equal Weight Universe",
            "S&P 500 Index Benchmark"
        ])
    ]
    .sort_values("Balanced Score")
    .groupby("method", as_index=False)
    .head(1)
)

best_momentum = momentum_metrics.sort_values(
    ["Daily Sharpe", "Annualized Return"],
    ascending=[False, False]
).head(1)

momentum_comparison = pd.concat([best_core_by_method, best_momentum], ignore_index=True, sort=False)

safe_to_csv(momentum_comparison, OUTPUT_DIR / "step5_2020_momentum_comparison_table.csv", index=False)
write_text(
    OUTPUT_DIR / "step5_2020_momentum_comparison_table.md",
    df_to_markdown_simple(momentum_comparison)
)


# ============================================================
# 13. EXPERIMENT 7.3 — RANDOM REPRESENTATIVE-DATE BENCHMARK
# ============================================================

random_trial_metrics_rows = []
random_nav_series = []
random_return_series = []
random_rebalance_logs = []

RANDOM_H = 21
RANDOM_L = 0.20
RANDOM_K = 3

for seed_i in range(RANDOM_DATE_TRIALS):
    seed = RANDOM_SEED_BASE + seed_i
    strategy_name = (
        f"Random Representative-Date, 2020-start universe | "
        f"seed={seed} | h={RANDOM_H} | l={RANDOM_L:.2f} | k={RANDOM_K}"
    )

    if seed_i % 10 == 0:
        print(f"Running random-date trial {seed_i + 1}/{RANDOM_DATE_TRIALS}")

    res = backtest_strategy(
        stock_returns=stock_returns,
        strategy_type="Random Representative-Date",
        method_label=strategy_name,
        h=RANDOM_H,
        l=RANDOM_L,
        k=RANDOM_K,
        W=W,
        random_seed=seed,
    )

    row = res["metrics"].copy()
    row["seed"] = seed
    random_trial_metrics_rows.append(row)

    random_nav_series.append(res["nav"])
    random_return_series.append(res["daily_returns"])

    rlog = res["rebalance_log"].copy()
    rlog["seed"] = seed
    random_rebalance_logs.append(rlog)

random_trials_metrics = pd.DataFrame(random_trial_metrics_rows)
random_daily_nav = pd.concat(random_nav_series, axis=1)
random_daily_returns = pd.concat(random_return_series, axis=1)
random_rebalance_log = pd.concat(random_rebalance_logs, ignore_index=True)

safe_to_csv(random_trials_metrics, OUTPUT_DIR / "step5_2020_random_date_trials_metrics.csv", index=False)
safe_to_csv(random_daily_nav, OUTPUT_DIR / "step5_2020_random_date_daily_nav.csv", index=True)
safe_to_csv(random_daily_returns, OUTPUT_DIR / "step5_2020_random_date_daily_returns.csv", index=True)
safe_to_csv(random_rebalance_log, OUTPUT_DIR / "step5_2020_random_date_rebalance_log.csv", index=False)

random_metric_cols = [
    "Cumulative Return",
    "Annualized Return",
    "Daily Sharpe",
    "Daily Max Drawdown",
    "Average Turnover",
    "Average Number of Stocks",
]

summary_rows = []
percentile_rows = []

for col in random_metric_cols:
    s = random_trials_metrics[col].dropna()

    summary_rows.append({
        "metric": col,
        "mean": s.mean(),
        "median": s.median(),
        "std": s.std(ddof=1),
        "min": s.min(),
        "max": s.max(),
    })

    percentile_rows.append({
        "metric": col,
        "p05": s.quantile(0.05),
        "p25": s.quantile(0.25),
        "p50": s.quantile(0.50),
        "p75": s.quantile(0.75),
        "p95": s.quantile(0.95),
    })

random_summary = pd.DataFrame(summary_rows)
random_percentiles = pd.DataFrame(percentile_rows)

safe_to_csv(random_summary, OUTPUT_DIR / "step5_2020_random_date_summary.csv", index=False)
safe_to_csv(random_percentiles, OUTPUT_DIR / "step5_2020_random_date_percentiles.csv", index=False)

best_pca_2020 = ranking[ranking["method"] == "PCA-KMeans"].sort_values("Balanced Score").head(1)
best_mapper_2020 = ranking[ranking["method"] == "Mapper"].sort_values("Balanced Score").head(1)


def compare_random_to_target(target_df, target_name):
    rows = []

    if target_df.empty:
        return pd.DataFrame()

    target = target_df.iloc[0]

    for col in random_metric_cols:
        random_s = random_trials_metrics[col].dropna()

        rows.append({
            "target": target_name,
            "metric": col,
            "target_value": target[col],
            "random_mean": random_s.mean(),
            "random_median": random_s.median(),
            "random_p05": random_s.quantile(0.05),
            "random_p95": random_s.quantile(0.95),
            "target_minus_random_mean": target[col] - random_s.mean(),
            "target_percentile_vs_random": (random_s <= target[col]).mean(),
        })

    return pd.DataFrame(rows)


random_vs_pca = compare_random_to_target(best_pca_2020, "Best PCA-KMeans 2020 configuration")
random_vs_mapper = compare_random_to_target(best_mapper_2020, "Best Mapper 2020 configuration")

safe_to_csv(random_vs_pca, OUTPUT_DIR / "step5_2020_random_date_vs_pca_kmeans.csv", index=False)
safe_to_csv(random_vs_mapper, OUTPUT_DIR / "step5_2020_random_date_vs_mapper.csv", index=False)


# ============================================================
# 14. EXPERIMENT 7.4 — BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def moving_block_bootstrap_indices(n, block_length, rng):
    idx = []

    while len(idx) < n:
        start = rng.integers(0, max(1, n - block_length + 1))
        block = list(range(start, min(start + block_length, n)))
        idx.extend(block)

    return np.array(idx[:n])


def metrics_from_bootstrap_returns(returns):
    returns = pd.Series(returns).dropna()

    if len(returns) == 0:
        return {
            "Annualized Return": np.nan,
            "Daily Sharpe": np.nan,
            "Daily Max Drawdown": np.nan,
        }

    nav = INITIAL_NAV * (1.0 + returns).cumprod()
    cumulative_return = nav.iloc[-1] / INITIAL_NAV - 1.0

    return {
        "Annualized Return": annualized_return_from_total_return(cumulative_return, len(returns)),
        "Daily Sharpe": sharpe_from_returns(returns),
        "Daily Max Drawdown": max_drawdown_from_nav_with_initial(nav, INITIAL_NAV),
    }


def get_series_by_strategy_name(name):
    if name in core_daily_returns.columns:
        return core_daily_returns[name].dropna()
    if name in momentum_daily_returns.columns:
        return momentum_daily_returns[name].dropna()
    raise KeyError(f"Strategy return series not found: {name}")


bootstrap_strategy_rows = []

if not best_pca_2020.empty:
    bootstrap_strategy_rows.append(best_pca_2020.iloc[0])

if not best_mapper_2020.empty:
    bootstrap_strategy_rows.append(best_mapper_2020.iloc[0])

best_ew_2020 = ranking[ranking["method"] == "Equal Weight Universe"].sort_values("Balanced Score").head(1)

if not best_ew_2020.empty:
    bootstrap_strategy_rows.append(best_ew_2020.iloc[0])

best_sp500_2020 = ranking[ranking["method"] == "S&P 500 Index Benchmark"].sort_values("Balanced Score").head(1)

if not best_sp500_2020.empty:
    bootstrap_strategy_rows.append(best_sp500_2020.iloc[0])

if not best_momentum.empty:
    bootstrap_strategy_rows.append(best_momentum.iloc[0])

bootstrap_strategies = (
    pd.DataFrame(bootstrap_strategy_rows)
    .drop_duplicates(subset=["strategy"])
    .reset_index(drop=True)
)

bootstrap_rng = np.random.default_rng(RANDOM_SEED_BASE + 999)

bootstrap_metric_records = []
bootstrap_sample_store = {}

for _, row in bootstrap_strategies.iterrows():
    strategy_name = row["strategy"]
    returns = get_series_by_strategy_name(strategy_name).dropna()

    values = {
        "Annualized Return": [],
        "Daily Sharpe": [],
        "Daily Max Drawdown": [],
    }

    r_values = returns.values
    n = len(r_values)

    if n < BOOTSTRAP_BLOCK_LENGTH:
        raise RuntimeError(f"Too few returns for bootstrap: {strategy_name}")

    for b in range(BOOTSTRAP_SAMPLES):
        idx = moving_block_bootstrap_indices(n, BOOTSTRAP_BLOCK_LENGTH, bootstrap_rng)
        sampled = r_values[idx]
        m = metrics_from_bootstrap_returns(sampled)

        for metric_name in values:
            values[metric_name].append(m[metric_name])

    bootstrap_sample_store[strategy_name] = pd.DataFrame(values)

    for metric_name, vals in values.items():
        s = pd.Series(vals).dropna()

        bootstrap_metric_records.append({
            "strategy": strategy_name,
            "method": row.get("method", None),
            "metric": metric_name,
            "bootstrap_samples": BOOTSTRAP_SAMPLES,
            "block_length": BOOTSTRAP_BLOCK_LENGTH,
            "mean": s.mean(),
            "median": s.median(),
            "std": s.std(ddof=1),
            "ci_2_5": s.quantile(0.025),
            "ci_5": s.quantile(0.05),
            "ci_95": s.quantile(0.95),
            "ci_97_5": s.quantile(0.975),
        })

bootstrap_ci_metrics = pd.DataFrame(bootstrap_metric_records)
safe_to_csv(bootstrap_ci_metrics, OUTPUT_DIR / "step5_2020_bootstrap_ci_metrics.csv", index=False)

pairwise_records = []

pca_strategy_names = [s for s in bootstrap_sample_store.keys() if "PCA-KMeans" in s]

if pca_strategy_names:
    pca_name = pca_strategy_names[0]
    pca_samples = bootstrap_sample_store[pca_name]

    for other_name, other_samples in bootstrap_sample_store.items():
        if other_name == pca_name:
            continue

        for metric_name in ["Annualized Return", "Daily Sharpe", "Daily Max Drawdown"]:
            diff = pca_samples[metric_name].values - other_samples[metric_name].values
            diff = pd.Series(diff).dropna()

            pairwise_records.append({
                "comparison": f"PCA-KMeans minus {other_name}",
                "pca_strategy": pca_name,
                "other_strategy": other_name,
                "metric": metric_name,
                "bootstrap_samples": BOOTSTRAP_SAMPLES,
                "block_length": BOOTSTRAP_BLOCK_LENGTH,
                "mean_difference": diff.mean(),
                "median_difference": diff.median(),
                "std_difference": diff.std(ddof=1),
                "ci_2_5": diff.quantile(0.025),
                "ci_5": diff.quantile(0.05),
                "ci_95": diff.quantile(0.95),
                "ci_97_5": diff.quantile(0.975),
                "ci_95_contains_zero": bool(diff.quantile(0.025) <= 0 <= diff.quantile(0.975)),
            })

bootstrap_pairwise = pd.DataFrame(pairwise_records)
safe_to_csv(bootstrap_pairwise, OUTPUT_DIR / "step5_2020_bootstrap_pairwise_differences.csv", index=False)

bootstrap_methodology = {
    "bootstrap_type": "moving block bootstrap",
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "block_length": BOOTSTRAP_BLOCK_LENGTH,
    "metrics": ["Annualized Return", "Daily Sharpe", "Daily Max Drawdown"],
    "pairwise_reference": "Best PCA-KMeans 2020 configuration by Step 5 Balanced Score",
    "random_seed_base": RANDOM_SEED_BASE,
}

safe_to_json(bootstrap_methodology, OUTPUT_DIR / "step5_2020_bootstrap_methodology.json")

write_text(
    OUTPUT_DIR / "step5_2020_bootstrap_summary.md",
    "# Step 5 2020 Universe Bootstrap Summary\n\n"
    f"- Bootstrap type: moving block bootstrap\n"
    f"- Bootstrap samples: {BOOTSTRAP_SAMPLES}\n"
    f"- Block length: {BOOTSTRAP_BLOCK_LENGTH} trading days\n"
    "- Reference comparison: PCA-KMeans minus each selected comparator\n\n"
    "Interpretation guardrail: bootstrap intervals are descriptive uncertainty estimates. "
    "Do not claim statistical superiority unless the relevant interval supports it.\n"
)


# ============================================================
# 15. FIGURE — 2020 START-DATE UNIVERSE NAV COMPARISON
# ============================================================

plot_candidates = []

for df in [best_pca_2020, best_mapper_2020, best_ew_2020, best_sp500_2020, best_momentum]:
    if df is not None and len(df) > 0:
        plot_candidates.append(df.iloc[0]["strategy"])

plot_candidates = list(dict.fromkeys(plot_candidates))

plt.figure(figsize=(12, 7))

for name in plot_candidates:
    if name in core_daily_nav.columns:
        s = core_daily_nav[name].dropna()
    elif name in momentum_daily_nav.columns:
        s = momentum_daily_nav[name].dropna()
    else:
        continue

    if len(s) > 0:
        plt.plot(s.index, s.values, label=name)

plt.title("Step 5 — 2020 Start-Date Universe NAV Comparison")
plt.xlabel("Date")
plt.ylabel("NAV")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "nav_comparison_2020_universe.png", dpi=200)
plt.savefig(FIG_DIR / "nav_comparison_2020_universe.pdf")
plt.close()


# ============================================================
# 16. METRIC CONSISTENCY CHECKS
# ============================================================

consistency_rows = []

for r in core_results:
    consistency_rows.append(metric_consistency(r["strategy"], r["nav"], r["daily_returns"]))

for r in momentum_results:
    consistency_rows.append(metric_consistency(r["strategy"], r["nav"], r["daily_returns"]))

for col in random_daily_nav.columns:
    consistency_rows.append(metric_consistency(col, random_daily_nav[col], random_daily_returns[col]))

consistency_df = pd.DataFrame(consistency_rows)
safe_to_csv(consistency_df, OUTPUT_DIR / "step5_2020_metric_consistency_check.csv", index=False)

failed_consistency = consistency_df[consistency_df["result"] == "FAIL"]

if len(failed_consistency) > 0:
    safe_to_csv(failed_consistency, OUTPUT_DIR / "step5_2020_failed_metric_consistency_rows.csv", index=False)
    raise RuntimeError(
        "Metric consistency check failed. "
        "Saved failed rows to step5_2020_failed_metric_consistency_rows.csv. "
        f"First failed strategies: {failed_consistency['strategy'].tolist()[:10]}"
    )


# ============================================================
# 17. AUDIT SUMMARY AND RUN CONFIG
# ============================================================

clean_start_rows = []

for h in H_GRID:
    info = get_clean_rebalance_indices(stock_returns, h=h, W=W, anchor_date=ANCHOR_DATE, k=K_DEFAULT)

    clean_start_rows.append({
        "h": h,
        "raw_first_rebalance_date": str(info["raw_first_rebalance_date"].date()),
        "clean_evaluation_start_date": str(info["clean_evaluation_start_date"].date()),
        "skipped_initial_rebalance_count": info["skipped_initial_rebalance_count"],
        "reason_for_skipping_initial_invalid_rebalances": info["reason_for_skipping_initial_invalid_rebalances"],
    })

clean_start_df = pd.DataFrame(clean_start_rows)

# FIXED: correct argument order is safe_to_csv(df, path, index=False)
safe_to_csv(clean_start_df, META_DIR / "step5_2020_clean_evaluation_start_by_h.csv", index=False)

run_config = {
    "created_at": now_str(),
    "step": "Step 5 — 2020 Start-Date S&P 500 Universe Robustness Round",
    "variant": "Variant 7A — use pre-2020 price history for 504-day lookback; evaluation begins only after 2020 anchor date",
    "anchor_date": str(ANCHOR_DATE.date()),
    "evaluation_end_date": str(EVALUATION_END_DATE.date()),
    "price_start_date": str(PRICE_START_DATE.date()),
    "price_end_date_exclusive": str(PRICE_END_DATE_EXCLUSIVE.date()),
    "W": W,
    "k": K_DEFAULT,
    "h_grid": H_GRID,
    "l_grid": L_GRID,
    "transaction_cost_rate": TRANSACTION_COST_RATE,
    "initial_nav": INITIAL_NAV,
    "random_date_trials": RANDOM_DATE_TRIALS,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "bootstrap_block_length": BOOTSTRAP_BLOCK_LENGTH,
    "random_seed_base": RANDOM_SEED_BASE,
    "kmeans_random_state": KMEANS_RANDOM_STATE,
    "kmeans_n_init": KMEANS_N_INIT,
    "mapper_params": {
        "cover_intervals": MAPPER_COVER_INTERVALS,
        "overlap": MAPPER_OVERLAP,
        "local_clusterer": "DBSCAN",
        "dbscan_min_samples": MAPPER_DBSCAN_MIN_SAMPLES,
    },
    "global_dbscan_params": {
        "min_samples": GLOBAL_DBSCAN_MIN_SAMPLES,
        "eps_rule": "70th percentile of kNN distance with KMeans fallback if no non-noise cluster",
    },
    "ticker_counts": {
        "raw_ticker_count": raw_ticker_count,
        "normalized_unique_ticker_count": normalized_unique_ticker_count,
        "available_ticker_count": available_ticker_count,
        "missing_unavailable_ticker_count": missing_unavailable_ticker_count,
    },
    "clean_evaluation_start_by_h": clean_start_rows,
    "data_source": {
        "universe_file": "sp500_constituents_2020_start.csv",
        "universe_source": "fja05680/sp500 GitHub historical components as provided in input CSV",
        "stock_price_source": "yfinance auto_adjust=True",
        "benchmark": "^GSPC yfinance auto_adjust=True",
    },
    "paper_guardrail": (
        "This is not fully survivorship-free. It is a fixed start-2020 universe robustness check "
        "using a public reconstructed 2020 start-date S&P 500 universe."
    ),
}

safe_to_json(run_config, META_DIR / "run_config.json")

data_sources = {
    "sp500_constituents_2020_start.csv": {
        "description": "S&P 500 constituent list fixed around 2020 start date",
        "source": "fja05680/sp500 GitHub historical components, as provided by input artifact",
        "guardrail": "public reconstructed universe, not official CRSP/WRDS point-in-time membership",
    },
    "stock_prices": {
        "source": "yfinance",
        "auto_adjust": True,
        "start": str(PRICE_START_DATE.date()),
        "end_exclusive": str(PRICE_END_DATE_EXCLUSIVE.date()),
    },
    "sp500_benchmark": {
        "ticker": "^GSPC",
        "source": "yfinance",
        "auto_adjust": True,
    },
}

safe_to_json(data_sources, META_DIR / "data_sources.json")

audit_md = f"""# Step 5 Audit Summary — 2020 Start-Date Universe Robustness

## Scope

This output implements Step 5 / Experiment 7 from `plan.md`.

The experiment uses a S&P 500 constituent universe fixed around the start of 2020 and evaluates strategies only from 2020 onward.

## Variant

Variant 7A is used:

- Universe fixed as of start 2020.
- Pre-2020 price history is used only for the 504-trading-day lookback.
- Evaluation starts from the first valid rebalance date on or after 2020-01-01.

## Universe

- Source file: `sp500_constituents_2020_start.csv`
- Raw ticker count: {raw_ticker_count}
- Normalized unique ticker count: {normalized_unique_ticker_count}
- Available ticker count: {available_ticker_count}
- Missing/unavailable ticker count: {missing_unavailable_ticker_count}

Ticker-level availability is logged in:

`00_metadata/missing_or_unavailable_universe_tickers.csv`

## Clean Evaluation Start

{df_to_markdown_simple(clean_start_df)}

## Experiments Completed

### Experiment 7.1 — Core Representative-State Comparison

Methods:

- Mapper
- Global DBSCAN
- PCA-KMeans
- Equal Weight Universe
- S&P 500 Index Benchmark

Grid:

- h in {H_GRID}
- l in {L_GRID}
- k = {K_DEFAULT}
- W = {W}

### Experiment 7.2 — Momentum Benchmark

Momentum lookbacks:

- 21D
- 63D
- 126D

Grid:

- h in {H_GRID}
- l in {L_GRID}

### Experiment 7.3 — Random Representative-Date Benchmark

- h = {RANDOM_H}
- l = {RANDOM_L}
- k = {RANDOM_K}
- random trials = {RANDOM_DATE_TRIALS}

### Experiment 7.4 — Bootstrap Confidence Intervals

- Bootstrap type: moving block bootstrap
- Bootstrap samples: {BOOTSTRAP_SAMPLES}
- Block length: {BOOTSTRAP_BLOCK_LENGTH}

## Metric Consistency

Metric consistency check result:

- PASS count: {(consistency_df["result"] == "PASS").sum()}
- FAIL count: {(consistency_df["result"] == "FAIL").sum()}

All reported strategy metrics are computed from NAV-derived daily returns after transaction cost.

## Important Guardrail

This is not fully survivorship-free. It is a fixed start-2020 universe robustness check using a public reconstructed S&P 500 constituent list.

Do not claim:

- proven alpha generation;
- direct investability;
- market-beating strategy;
- TDA superiority;
- Mapper superiority;
- survivorship-free historical performance;
- statistical superiority unless supported by bootstrap or holdout evidence.
"""

write_text(OUTPUT_DIR / "step5_2020_audit_summary.md", audit_md)


# ============================================================
# 18. OUTPUT FILE CHECK AND HARD ARTIFACT GATE
# ============================================================

REQUIRED_OUTPUT_FILES = [
    "step5_2020_core_metrics.csv",
    "step5_2020_core_daily_nav.csv",
    "step5_2020_core_daily_returns.csv",
    "step5_2020_core_rebalance_log.csv",
    "step5_2020_core_balanced_score_ranking.csv",

    "step5_2020_momentum_metrics.csv",
    "step5_2020_momentum_daily_nav.csv",
    "step5_2020_momentum_daily_returns.csv",
    "step5_2020_momentum_rebalance_log.csv",
    "step5_2020_momentum_comparison_table.md",

    "step5_2020_random_date_trials_metrics.csv",
    "step5_2020_random_date_summary.csv",
    "step5_2020_random_date_percentiles.csv",
    "step5_2020_random_date_vs_pca_kmeans.csv",
    "step5_2020_random_date_vs_mapper.csv",

    "step5_2020_bootstrap_ci_metrics.csv",
    "step5_2020_bootstrap_pairwise_differences.csv",
    "step5_2020_bootstrap_methodology.json",
    "step5_2020_bootstrap_summary.md",

    "step5_2020_metric_consistency_check.csv",
    "step5_2020_audit_summary.md",

    "00_metadata/run_config.json",
    "00_metadata/data_sources.json",
    "00_metadata/step5_2020_universe_tickers.csv",
    "00_metadata/missing_or_unavailable_universe_tickers.csv",
    "00_metadata/downloaded_stock_prices_yfinance_auto_adjust_true.csv",
    "00_metadata/downloaded_stock_returns_yfinance_auto_adjust_true.csv",
    "00_metadata/downloaded_sp500_index_yfinance_auto_adjust_true.csv",
    "00_metadata/failed_yfinance_batches.csv",
    "00_metadata/step5_2020_clean_evaluation_start_by_h.csv",

    "figures/nav_comparison_2020_universe.png",
    "figures/nav_comparison_2020_universe.pdf",
]

artifact_rows = []

for rel in REQUIRED_OUTPUT_FILES:
    p = OUTPUT_DIR / rel

    artifact_rows.append({
        "required_file": rel,
        "exists": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else 0,
    })

artifact_check_df = pd.DataFrame(artifact_rows)
safe_to_csv(artifact_check_df, OUTPUT_DIR / "step5_2020_artifact_file_check.csv", index=False)

missing_required_outputs = artifact_check_df[~artifact_check_df["exists"]]["required_file"].tolist()

if missing_required_outputs:
    raise RuntimeError(
        "Missing required Step 5 output artifact(s). "
        f"Notebook/code is not allowed to finish. Missing: {missing_required_outputs}"
    )

if not (OUTPUT_DIR / "step5_2020_artifact_file_check.csv").exists():
    raise RuntimeError("Missing step5_2020_artifact_file_check.csv")

for _, row in clean_start_df.iterrows():
    if pd.Timestamp(row["clean_evaluation_start_date"]) < ANCHOR_DATE:
        raise RuntimeError(
            f"Clean evaluation start for h={row['h']} is before anchor date: "
            f"{row['clean_evaluation_start_date']}"
        )

if core_rebalance_log["strategy"].isna().any():
    bad = core_rebalance_log[core_rebalance_log["strategy"].isna()].head()
    raise RuntimeError(f"Core rebalance log contains null strategy rows:\n{bad}")

if momentum_rebalance_log["strategy"].isna().any():
    bad = momentum_rebalance_log[momentum_rebalance_log["strategy"].isna()].head()
    raise RuntimeError(f"Momentum rebalance log contains null strategy rows:\n{bad}")

required_missing_cols = {"raw_ticker", "normalized_ticker", "status", "reason"}

if not required_missing_cols.issubset(set(missing_unavailable_df.columns)):
    raise RuntimeError(
        "missing_or_unavailable_universe_tickers.csv schema is invalid. "
        f"Required columns: {required_missing_cols}"
    )

if (consistency_df["result"] == "FAIL").any():
    raise RuntimeError("Metric consistency has FAIL rows.")

zip_folder(OUTPUT_DIR, OUTPUT_ZIP)

if not OUTPUT_ZIP.exists() or OUTPUT_ZIP.stat().st_size == 0:
    raise RuntimeError("Output zip was not created correctly.")

print("\n================ STEP 5 COMPLETE ================")
print("Output folder:", OUTPUT_DIR)
print("Output zip:", OUTPUT_ZIP)

print("\nCore metrics preview:")
display(core_metrics.head())

print("\nCore balanced score ranking preview:")
display(ranking.head(10))

print("\nMomentum metrics preview:")
display(momentum_metrics.head())

print("\nRandom-date summary:")
display(random_summary)

print("\nBootstrap CI preview:")
display(bootstrap_ci_metrics.head())

print("\nArtifact check:")
display(artifact_check_df)

print("\nThe work is done.")

try:
    from IPython.display import Audio, display as ipy_display

    duration = 0.4
    freq = 880
    rate = 44100
    t = np.linspace(0, duration, int(rate * duration), False)
    tone = 0.2 * np.sin(freq * 2 * np.pi * t)

    ipy_display(Audio(tone, rate=rate, autoplay=True))
except Exception:
    pass

All required input files are present.
Working directory: D:\Notebook\Professorship\research
Found step1_experiment_metadata.json: D:\Notebook\Professorship\research\tda_step5_2020_universe_outputs\_extracted_inputs\experiment_input\experiment_input\01_step1_mapper\tda_experiment_outputs_step1\tda_experiment_outputs_step1\step1_experiment_metadata.json
INITIAL_NAV used: 1000000.0
Raw ticker count: 504
Normalized unique ticker count: 504


$ATVI: possibly delisted; no timezone found
$ADS: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$ALXN: possibly delisted; no timezone found
$ARNC: possibly delisted; no timezone found
$ABMD: possibly delisted; no timezone found
$ANTM: possibly delisted; no timezone found
$ANSS: possibly delisted; no timezone found
$BLL: possibly delisted; no timezone found
$APC: possibly delisted; no price data found  (1d 2017-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1483246800, endDate = 1767243600")
$ABC: possibly delisted; no timezone found
$BHGE: possibly delisted; no timezone found

12 Failed downloads:
['ATVI', 'ADS', 'AGN', 'ALXN', 'ARNC', 'ABMD', 'ANTM', 'ANSS', 'BLL', 'ABC', 'BHGE']: possibly delisted; no timezone found
['APC']: possibly delisted; no price data found  (1d 2017-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1483246800, endDate = 1767243600")


$CERN: possibly delisted; no timezone found
$DISCK: possibly delisted; no timezone found
$CTXS: possibly delisted; no timezone found
$CXO: possibly delisted; no timezone found
$DISH: possibly delisted; no timezone found
$CTL: possibly delisted; no timezone found
$DRE: possibly delisted; no timezone found
$CBS: possibly delisted; no timezone found
$DISCA: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found
$COG: possibly delisted; no timezone found
$CMA: possibly delisted; no timezone found
$DFS: possibly delisted; no timezone found
$DWDP: possibly delisted; no timezone found

14 Failed downloads:
['CERN', 'DISCK', 'CTXS', 'CXO', 'DISH', 'CTL', 'DRE', 'CBS', 'DISCA', 'CELG', 'COG', 'CMA', 'DFS', 'DWDP']: possibly delisted; no timezone found


$HFC: possibly delisted; no timezone found
$ETFC: possibly delisted; no timezone found
$FL: possibly delisted; no timezone found
$FBHS: possibly delisted; no timezone found
$HES: possibly delisted; no timezone found
$FLT: possibly delisted; no timezone found
$HRS: possibly delisted; no timezone found
$HCP: possibly delisted; no timezone found
$HOLX: possibly delisted; no timezone found
$HBI: possibly delisted; no timezone found
$FRC: possibly delisted; no timezone found
$GPS: possibly delisted; no timezone found
$FLIR: possibly delisted; no timezone found

13 Failed downloads:
['HFC', 'ETFC', 'FL', 'FBHS', 'HES', 'FLT', 'HRS', 'HCP', 'HOLX', 'HBI', 'FRC', 'GPS', 'FLIR']: possibly delisted; no timezone found


$IPG: possibly delisted; no timezone found
$LLL: possibly delisted; no timezone found
$JWN: possibly delisted; no timezone found
$JNPR: possibly delisted; no timezone found
$K: possibly delisted; no timezone found
$MMC: possibly delisted; no timezone found
$KSU: possibly delisted; no timezone found
$JEC: possibly delisted; no timezone found

8 Failed downloads:
['IPG', 'LLL', 'JWN', 'JNPR', 'K', 'MMC', 'KSU', 'JEC']: possibly delisted; no timezone found


...


Bootstrap CI preview:


,strategy,method,metric,bootstrap_samples,block_length,mean,median,std,ci_2_5,ci_5,ci_95,ci_97_5
0,"PCA-KMeans Representative-State, 2020-start un...",PCA-KMeans,Annualized Return,1000,21,0.124889,0.123965,0.093764,-0.046765,-0.026001,0.280263,0.300896
1,"PCA-KMeans Representative-State, 2020-start un...",PCA-KMeans,Daily Sharpe,1000,21,0.659362,0.650064,0.404029,-0.077823,0.010140,1.306457,1.457164
2,"PCA-KMeans Representative-State, 2020-start un...",PCA-KMeans,Daily Max Drawdown,1000,21,-0.364373,-0.359988,0.118506,-0.627065,-0.572928,-0.185596,-0.170657
3,"Mapper Representative-State, 2020-start univer...",Mapper,Annualized Return,1000,21,0.121159,0.123196,0.094158,-0.064606,-0.040902,0.274311,0.306905
4,"Mapper Representative-State, 2020-start univer...",Mapper,Daily Sharpe,1000,21,0.644301,0.648489,0.404225,-0.130544,-0.045018,1.310987,1.420642



Artifact check:


,required_file,exists,size_bytes
0,step5_2020_core_metrics.csv,True,14692
1,step5_2020_core_daily_nav.csv,True,948036
2,step5_2020_core_daily_returns.csv,True,1095531
3,step5_2020_core_rebalance_log.csv,True,279493
4,step5_2020_core_balanced_score_ranking.csv,True,15920
5,step5_2020_momentum_metrics.csv,True,11563
6,step5_2020_momentum_daily_nav.csv,True,778431
7,step5_2020_momentum_daily_returns.csv,True,898336
8,step5_2020_momentum_rebalance_log.csv,True,189344
9,step5_2020_momentum_comparison_table.md,True,3786



The work is done.
